# Bagging Classifier: Iris Dataset

## Overview

**Bagging** (Bootstrap Aggregating) is an ensemble method that builds multiple decision trees independently on different random samples of the training data, then averages or votes on their predictions. Bagging reduces variance and improves generalization.

## Key Concept: How Bagging Works

1. **Bootstrap Sampling:** Create multiple subsets by randomly sampling from the training data with replacement
2. **Train Multiple Models:** Train an independent decision tree on each subset
3. **Aggregate Predictions:**
   - **Classification:** Use majority voting (most frequent class)
   - **Regression:** Use averaging of predictions
4. **Result:** Lower variance than a single tree with similar bias

## Learning Objectives

By the end of this notebook, you will understand:
1. How Bagging reduces variance
2. How to train a Bagging Classifier
3. How to evaluate ensemble performance
4. The effect of number of estimators on performance

## Step 1: Import Libraries

In [ ]:
from matplotlib import pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

## Step 2: Load and Prepare Data

In [ ]:
# Load the Iris dataset
iris_ds = load_iris()
X = iris_ds.data
y = iris_ds.target

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Classes: {iris_ds.target_names}')

## Step 3: Train Single Decision Tree (Baseline)

First, we train a single decision tree to use as a baseline for comparison.

In [ ]:
# Single Decision Tree (baseline)
dt_single = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    random_state=42
)
dt_single.fit(X_train, y_train)

# Evaluate single tree
y_pred_single = dt_single.predict(X_test)
acc_single = accuracy_score(y_test, y_pred_single)

print(f'Single Decision Tree Accuracy: {acc_single:.4f}')

## Step 4: Train Bagging Classifier

Bagging trains multiple base estimators (decision trees) on bootstrap samples and aggregates their predictions. Key parameters:
- **n_estimators:** Number of base estimators (trees) in the ensemble
- **max_samples:** Number of samples to draw for each base estimator (default: all samples)
- **bootstrap:** Whether to use bootstrap sampling (with replacement)

In [ ]:
# Bagging Classifier with 10 decision trees
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(criterion='gini', max_depth=5),
    n_estimators=10,      # Number of trees
    max_samples=1.0,      # Use all samples for each tree
    bootstrap=True,       # Use bootstrap sampling (with replacement)
    random_state=42
)

# Train the ensemble
bagging_clf.fit(X_train, y_train)

# Evaluate
y_pred_bagging = bagging_clf.predict(X_test)
acc_bagging = accuracy_score(y_test, y_pred_bagging)

print(f'Bagging Classifier Accuracy (10 trees): {acc_bagging:.4f}')
print(f'Improvement over single tree: {(acc_bagging - acc_single):.4f}')

## Step 5: Effect of Number of Estimators

Let's see how the accuracy changes as we increase the number of trees in the ensemble.

In [ ]:
# Test with different numbers of estimators
n_estimators_list = [1, 5, 10, 20, 50, 100]
bagging_accuracies = []

for n_est in n_estimators_list:
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(criterion='gini', max_depth=5),
        n_estimators=n_est,
        bootstrap=True,
        random_state=42
    )
    bag.fit(X_train, y_train)
    y_pred = bag.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    bagging_accuracies.append(acc)
    print(f'n_estimators={n_est:3d}: Accuracy = {acc:.4f}')

## Step 6: Visualize Performance

In [ ]:
# Plot accuracy vs. number of estimators
plt.figure(figsize=(8, 5))
plt.plot(n_estimators_list, bagging_accuracies, 'o-', linewidth=2, markersize=8, label='Bagging')
plt.axhline(y=acc_single, color='r', linestyle='--', linewidth=2, label=f'Single Tree ({acc_single:.4f})')
plt.xlabel('Number of Estimators')
plt.ylabel('Test Accuracy')
plt.title('Bagging: Accuracy vs. Number of Trees')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 7: Detailed Evaluation

Let's evaluate our final bagging classifier with detailed metrics.

In [ ]:
# Use the best performing model (10 estimators)
final_bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(criterion='gini', max_depth=5),
    n_estimators=10,
    bootstrap=True,
    random_state=42
)
final_bagging.fit(X_train, y_train)
y_pred_final = final_bagging.predict(X_test)

# Classification report
print('Classification Report:')
print(classification_report(y_test, y_pred_final, target_names=iris_ds.target_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_final)
print('\nConfusion Matrix:')
print(cm)

## Key Takeaways

1. **Bagging reduces variance** by training multiple trees on different samples and aggregating predictions
2. **Parallel training:** Unlike boosting, bagging trees are trained independently and can be parallelized
3. **Bootstrap sampling:** Each tree sees a different random subset of the training data
4. **Voting aggregation:** For classification, predictions are combined via majority voting
5. **Diminishing returns:** Performance typically plateaus after ~10-50 trees depending on the dataset
6. **Better than single tree:** Bagging usually provides better test accuracy than a single decision tree